# Laboratorio 6 — Análisis de redes sociales (YouTube)


## 1. Carga, comprensión e integración de los datos


### 1.1. Carga de los archivos

Se usa el encoding `utf-8-sig`
para evitar que ese carácter quede pegado al nombre de la primera columna (`video_id`).


In [1]:
%pip install simplemma
%pip install nltk

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

import re
import urllib.parse

import emoji
import simplemma
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

STOPWORDS_ES = set(stopwords.words("spanish"))
print(f"Stopwords en español cargadas (nltk): {len(STOPWORDS_ES)}")


# utf-8-sig evita que el BOM (\ufeff) quede pegado al nombre de la primera columna
videos = pd.read_csv("data/youtube_videos.csv", encoding="utf-8-sig")
comments = pd.read_csv("data/youtube_comments.csv", encoding="utf-8-sig")

print(f"youtube_videos.csv   -> {videos.shape[0]} filas, {videos.shape[1]} columnas")
print(f"youtube_comments.csv -> {comments.shape[0]} filas, {comments.shape[1]} columnas")


Stopwords en español cargadas (nltk): 313
youtube_videos.csv   -> 293 filas, 20 columnas
youtube_comments.csv -> 406 filas, 17 columnas


In [3]:
videos.head(3)


,video_id,title,channel_name,channel_id,source_query,source_group,dataset_sources,channel_handle,published_time,view_count_text,description_snippet,video_url,query_hits,keywords,description,view_count,publish_date,upload_date,category,owner_handle
0,-5puKGEqcUc,INSIVUMEH pronostica incremento de lluvias par...,T13 Noticias Guatemala,UCq0Cm-3SKthEySQc2JZBi1A,guatemala lluvias,topic,youtube_guatemala.csv | youtube_guatemala_lab....,/@T13NoticiasGuatemala,hace 2 días,"2,390 vistas",El Departamento de Pronóstico de INSIVUMEH pre...,https://www.youtube.com/watch?v=-5puKGEqcUc,"[""guatemala lluvias""]","[""Canícula prolongada"", ""Chapin tv"", ""Fenómeno...",El Departamento de Pronóstico de INSIVUMEH pre...,2357,2026-08-28T22:00:20-07:00,2026-08-28T22:00:20-07:00,News & Politics,/@T13NoticiasGuatemala
1,-E7OPOLjMug,BERNARDO ARÉVALO CALIFICA CAMBIO EN EL MP COMO...,IDocumenta,UCgItjn_ZWFWcv1MlpKIkqgQ,@GobiernodelaRepublicadeGuatema,topic,youtube_target_channels.csv,/@iDocumenta,hace 3 meses,4 vistas,"El presidente de Guatemala, Bernardo Arévalo, ...",https://www.youtube.com/watch?v=-E7OPOLjMug,"[""@GobiernodelaRepublicadeGuatema""]",[],"El presidente de Guatemala, Bernardo Arévalo, ...",4,2026-05-13T16:00:03-07:00,2026-05-13T16:00:03-07:00,People & Blogs,/@iDocumenta
2,-KDglrIzRKo,¡HISTÓRICO! Mexico recupera petróleo robado po...,México Poder,UC-DpoeBbCMOMOH1-j71KPqA,guatemala noticias,topic,youtube_guatemala.csv | youtube_guatemala_lab....,/@M%C3%A9xicoPoder,hace 1 día,"29,736 vistas",México #PetróleoMexicano #SoberaníaEnergética ...,https://www.youtube.com/watch?v=-KDglrIzRKo,"[""guatemala noticias""]","[""Mexico recupera petroleo"", ""petroleo robado ...",#México #PetróleoMexicano #SoberaníaEnergética...,29736,2026-08-29T17:00:38-07:00,2026-08-29T17:00:38-07:00,People & Blogs,/@M%C3%A9xicoPoder


In [4]:
comments.head(3)


,video_id,comment_id,video_title,channel_name,channel_id,author_name,author_channel_id,text,source_query,source_group,dataset_sources,author_handle,published_text,like_count_text,reply_count,is_pinned,viewer_rating
0,j43HgwYFKfk,Ugw-J65a1iYL9hqhELh4AaABAg,La cooptación de Walter Mazariegos en la USAC,Quorum,UCE4rsXcgDb6e1-a9iTbWzfg,@MarcosCarillo-b1r,UCdFlugHJJa4l3YqWuNRmvXw,Ese corrupto amigo de la vieja fiscal los teng...,@quorumgt,topic,j43HgwYFKfk_out_comments.csv | quorum_complete...,/@MarcosCarillo-b1r,hace 6 meses,,0,False,NaN
1,06mFNPU0aB8,Ugw-ZT9t9wU2V-tCaUZ4AaABAg,Capturan a presuntos delincuentes disfrazados ...,Noti7,UCVpSRoZgngfSL03Nlbjtq9A,@RaulPerez-cw2vi,UCvl1tzQeBeGy6efPTRJXSCw,"Están jóvenes porque no buscan un trabajo, tu...",guatemala noticias,topic,youtube_guatemala_comments.csv | youtube_guate...,/@RaulPerez-cw2vi,hace 2 semanas,,0,False,NaN
2,j43HgwYFKfk,Ugw0xaOb2CYXXoudtwJ4AaABAg,La cooptación de Walter Mazariegos en la USAC,Quorum,UCE4rsXcgDb6e1-a9iTbWzfg,@iamjimalesssa,UCRAquv8el-tQ30bN7MlmySQ,Me dejaron con ganas de demandar la ilegalidad...,@quorumgt,topic,j43HgwYFKfk_out_comments.csv | quorum_complete...,/@iamjimalesssa,hace 1 año,4,0,False,NaN


### 1.2. Unidad de observación, llave primaria y variables relevantes

**`youtube_videos.csv`**
- **Unidad de observación:** un video publicado en YouTube (recuperado mediante una consulta de búsqueda, un canal
  o una fuente oficial de gobierno).
- **Llave primaria:** `video_id`. Se verifica abajo que es única en las 293 filas, por lo que identifica de forma
  inequívoca cada fila.
- **Variables relevantes:**
  - Identificadores/relacionales: `video_id`, `channel_id`, `channel_handle`, `owner_handle`.
  - Contenido para análisis de texto/temas: `title`, `description`, `description_snippet`, `keywords`, `category`.
  - Popularidad: `view_count` y `view_count_text`
  - Procedencia/muestreo: `source_query`, `source_group`, `query_hits`, `dataset_sources`
  - Temporales: `publish_date` / `upload_date` (fecha exacta ISO 8601) vs. `published_time` (texto relativo, depende
    del momento de recolección).

**`youtube_comments.csv`**
- **Unidad de observación:** un comentario principal publicado en un video de YouTube.
- **Llave primaria:** `comment_id`. Se verifica que es único en las 406 filas.
- **Llave foránea:** `video_id`, que permite relacionar cada comentario con su video en `youtube_videos.csv`.
- **Variables relevantes:**
  - Identificador del autor: `author_channel_id`.
  - Contenido: `text` — variable principal para tópicos, sentimiento y palabras frecuentes.
  - Interacción: `like_count_text` (texto, requiere limpieza) y `reply_count` (numérica; **no** identifica a los
    autores de las respuestas, solo cuenta cuántas hubo).
  - Procedencia: `source_query`, `source_group`, `dataset_sources`.
  - Variables sin utilidad observada: `is_pinned` (constante en `False`) y `viewer_rating` (100% vacía).


In [5]:
# Confirmación de llaves primarias (deben ser únicas)
print("video_id únicos en videos:     ", videos["video_id"].nunique(), "/", len(videos),
      "-> llave primaria válida:", videos["video_id"].is_unique)
print("comment_id únicos en comments: ", comments["comment_id"].nunique(), "/", len(comments),
      "-> llave primaria válida:", comments["comment_id"].is_unique)


video_id únicos en videos:      293 / 293 -> llave primaria válida: True
comment_id únicos en comments:  406 / 406 -> llave primaria válida: True


In [6]:
print("=== youtube_videos.csv: tipos de dato ===")
print(videos.dtypes)
print()
print("=== youtube_comments.csv: tipos de dato ===")
print(comments.dtypes)


=== youtube_videos.csv: tipos de dato ===
video_id               object
title                  object
channel_name           object
channel_id             object
source_query           object
source_group           object
dataset_sources        object
channel_handle         object
published_time         object
view_count_text        object
description_snippet    object
video_url              object
query_hits             object
keywords               object
description            object
view_count              int64
publish_date           object
upload_date            object
category               object
owner_handle           object
dtype: object

=== youtube_comments.csv: tipos de dato ===
video_id              object
comment_id            object
video_title           object
channel_name          object
channel_id            object
author_name           object
author_channel_id     object
text                  object
source_query          object
source_group          object
dataset_s

In [7]:
# Cardinalidad de las entidades principales que participan en la red social
print("Canales únicos (channel_id) en videos:         ", videos["channel_id"].nunique())
print("Videos únicos (video_id):                      ", videos["video_id"].nunique())
print("Autores únicos (author_channel_id) en comments: ", comments["author_channel_id"].nunique())
print("Comentarios únicos (comment_id):                ", comments["comment_id"].nunique())
print("Categorías únicas (category) en videos:         ", videos["category"].nunique())
print("Consultas de búsqueda únicas (source_query) en videos:  ", videos["source_query"].nunique())


Canales únicos (channel_id) en videos:          97
Videos únicos (video_id):                       293
Autores únicos (author_channel_id) en comments:  332
Comentarios únicos (comment_id):                 406
Categorías únicas (category) en videos:          11
Consultas de búsqueda únicas (source_query) en videos:   21


### 1.3. Relación entre canal, video, autor del comentario, comentario, categoría y consulta de búsqueda

- **Canal → Video (1\:N):** un canal (`channel_id`) publica muchos videos. `channel_id` es el identificador
  estable; `channel_name` es solo la etiqueta visible y puede repetirse o cambiar en el tiempo.
- **Video → Comentario (1\:N):** un video (`video_id`) puede recibir muchos comentarios principales. `video_id` es
  la llave que conecta ambos archivos.
- **Comentario → Autor (N\:1):** cada comentario tiene un único autor (`author_channel_id`), pero un mismo autor
  puede publicar comentarios en varios videos.
- **Video → Categoría (N\:1):** `category` es un atributo del video, no una entidad de red independiente; varios videos comparten la misma categoría.
- **Video → Consulta de búsqueda (N\:N):** `source_query`/`query_hits` no describen el tema del video sino el
  **procedimiento de muestreo**: un video puede haber sido recuperado por más de una consulta, y una misma consulta recupera muchos videos. `source_group` (`topic`, `official_gov`, `channel`) resume la
  estrategia de búsqueda utilizada.
- **Importante:** en `youtube_comments.csv`, `channel_id`/`channel_name` identifican al **canal dueño del video
  comentado**, no al autor del comentario. El autor del comentario es una entidad distinta, identificada por
  `author_channel_id`, que normalmente no coincide con el canal ni con el dueño del video.


In [8]:
check = comments.merge(
    videos[["video_id", "channel_id"]],
    on="video_id",
    how="left",
    suffixes=("_comment", "_video"),
)
inconsistentes = (check["channel_id_comment"] != check["channel_id_video"]).sum()
print(f"Comentarios donde channel_id no coincide con el channel_id del video: {inconsistentes} / {len(check)}")


Comentarios donde channel_id no coincide con el channel_id del video: 0 / 406


### 1.4. Integración de los conjuntos de datos mediante `video_id`


In [9]:
merged = comments.merge(
    videos,
    on="video_id",
    how="left",
    suffixes=("_comment", "_video"),
)

comentarios_con_video = merged["title"].notna().sum()  # 'title' solo existe si hubo match con youtube_videos
print(f"Comentarios totales:                         {len(comments)}")
print(f"Comentarios asociados exitosamente a un video: {comentarios_con_video} ({comentarios_con_video/len(comments):.1%})")
print(f"Comentarios sin video asociado (video_id huérfano): {len(comments) - comentarios_con_video}")


Comentarios totales:                         406
Comentarios asociados exitosamente a un video: 406 (100.0%)
Comentarios sin video asociado (video_id huérfano): 0


In [10]:
videos_con_comentarios = comments["video_id"].nunique()
videos_totales = videos["video_id"].nunique()
videos_sin_comentarios = videos_totales - videos_con_comentarios

print(f"Videos totales en el catálogo:        {videos_totales}")
print(f"Videos que recibieron algún comentario: {videos_con_comentarios} ({videos_con_comentarios/videos_totales:.1%})")
print(f"Videos sin comentarios en la muestra:   {videos_sin_comentarios} ({videos_sin_comentarios/videos_totales:.1%})")
print()
print("Top 10 videos con más comentarios en la muestra:")
print(comments.groupby("video_id").size().sort_values(ascending=False).head(10))


Videos totales en el catálogo:        293
Videos que recibieron algún comentario: 19 (6.5%)
Videos sin comentarios en la muestra:   274 (93.5%)

Top 10 videos con más comentarios en la muestra:
video_id
n8iP75gIpmw    161
j43HgwYFKfk     50
6W4u8sGEnGM     45
OkXlHx0hx-8     25
PjmxCj-a9Hg     25
lj983NWyAQY     25
yLZS3JiEBg8     16
06mFNPU0aB8     14
ndAZjHqzzT8     12
b_Hkrkl3adA      7
dtype: int64


**Resultado de la integración:** los 406 comentarios (100%) pudieron asociarse correctamente con un video mediante
`video_id`; no existen comentarios "huérfanos" (con un `video_id` ausente en `youtube_videos.csv`).

Sin embargo, la relación es muy asimétrica: de los **293 videos** del catálogo, solo **19 (6.5%)** recibieron al
menos un comentario en esta muestra; los **274 restantes (93.5%)** no tienen ningún comentario asociado. Además,
dentro de esos 19 videos la participación está muy concentrada: un solo video concentra 161 de los 406 comentarios
(~40%).


## 2. Calidad, limpieza y preprocesamiento

### 2.1. Diagnóstico inicial de calidad


In [11]:
print(f"videos:   {videos.shape[0]} filas x {videos.shape[1]} columnas")
print(f"comments: {comments.shape[0]} filas x {comments.shape[1]} columnas")
print()
print("Tipos de dato (videos):")
print(videos.dtypes.value_counts())
print()
print("Tipos de dato (comments):")
print(comments.dtypes.value_counts())


videos:   293 filas x 20 columnas
comments: 406 filas x 17 columnas

Tipos de dato (videos):
object    19
int64      1
Name: count, dtype: int64

Tipos de dato (comments):
object     14
int64       1
bool        1
float64     1
Name: count, dtype: int64


In [12]:
print("=== Valores faltantes por columna (solo columnas con NA > 0) ===")
print("\nvideos:")
print(videos.isna().sum().loc[lambda s: s > 0])
print("\ncomments:")
print(comments.isna().sum().loc[lambda s: s > 0])


=== Valores faltantes por columna (solo columnas con NA > 0) ===

videos:
published_time         13
view_count_text        13
description_snippet    25
description            26
dtype: int64

comments:
viewer_rating    406
dtype: int64


In [13]:
print("=== Duplicados ===")
print("Filas 100% duplicadas en videos:  ", videos.duplicated().sum())
print("Filas 100% duplicadas en comments:", comments.duplicated().sum())
print("video_id duplicado en videos:     ", videos['video_id'].duplicated().sum())
print("comment_id duplicado en comments: ", comments['comment_id'].duplicated().sum())
print()
print("Comentarios con 'text' exactamente duplicado (posible copy-paste o spam):",
      comments['text'].duplicated(keep=False).sum())
comments.loc[comments['text'].duplicated(keep=False), ['comment_id', 'video_id', 'author_channel_id', 'text']]


=== Duplicados ===
Filas 100% duplicadas en videos:   0
Filas 100% duplicadas en comments: 0
video_id duplicado en videos:      0
comment_id duplicado en comments:  0

Comentarios con 'text' exactamente duplicado (posible copy-paste o spam): 4


,comment_id,video_id,author_channel_id,text
112,Ugx74OPnWkWaFjpFuA94AaABAg.A7ki-1ED5b2A7nIcG2_svP,n8iP75gIpmw,UCFjudnPLoaSL1BaRuLahyHg,Cierto.
116,Ugx8FxDKPAY-M_JhRgd4AaABAg,n8iP75gIpmw,UCi1b_TRQfgO1Z9O2gNGGWNA,Ahora se destapo la CORRUPCION Q EXISTIA Y AHO...
272,Ugy__nsWCBggw9EzOUd4AaABAg.A7lbRpGUZoHA7nJxaD37u-,n8iP75gIpmw,UCFjudnPLoaSL1BaRuLahyHg,Cierto.
324,Ugz7WPxTOQEwmZpnx_V4AaABAg,n8iP75gIpmw,UCi1b_TRQfgO1Z9O2gNGGWNA,Ahora se destapo la CORRUPCION Q EXISTIA Y AHO...


In [14]:
print("=== Variables constantes (0 o 1 valor único) ===")
for nombre_df, df in [("videos", videos), ("comments", comments)]:
    for col in df.columns:
        if df[col].nunique(dropna=False) <= 1:
            print(f"{nombre_df}.{col}: único valor observado -> {df[col].unique()}")


=== Variables constantes (0 o 1 valor único) ===
comments.is_pinned: único valor observado -> [False]
comments.viewer_rating: único valor observado -> [nan]


In [15]:
def resumen_outliers_iqr(serie, nombre):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_superior = q3 + 1.5 * iqr
    n_outliers = (serie > limite_superior).sum()
    print(f"{nombre}: Q1={q1:.0f}  Q3={q3:.0f}  límite superior IQR={limite_superior:.0f}  "
          f"-> {n_outliers} valores por encima ({n_outliers/len(serie):.1%})")

resumen_outliers_iqr(videos["view_count"], "view_count")
print(videos["view_count"].describe())


view_count: Q1=215  Q3=7465  límite superior IQR=18340  -> 49 valores por encima (16.7%)
count    2.930000e+02
mean     6.043009e+04
std      5.157983e+05
min      2.000000e+00
25%      2.150000e+02
50%      1.175000e+03
75%      7.465000e+03
max      8.190449e+06
Name: view_count, dtype: float64


In [16]:
# Consistencia entre identificadores, nombres y handles
print("1) ¿Un mismo channel_id tiene siempre el mismo channel_name?")
inconsistentes_canal = (videos.groupby("channel_id")["channel_name"].nunique() > 1).sum()
print(f"   channel_id con más de un channel_name distinto: {inconsistentes_canal} / {videos['channel_id'].nunique()}")

print()
print("2) ¿owner_handle coincide con channel_handle? (según el enunciado, deberían coincidir 100%)")
print(f"   Coincidencia: {(videos['owner_handle'] == videos['channel_handle']).mean():.1%}")

print()
print("3) ¿upload_date coincide con publish_date?")
print(f"   Coincidencia: {(videos['upload_date'] == videos['publish_date']).mean():.1%}")

print()
print("4) ¿El channel_id reportado en comments coincide con el channel_id del video (ya verificado en 1.3)?")
chk = comments.merge(videos[["video_id", "channel_id"]], on="video_id", suffixes=("_comment", "_video"))
print(f"   Inconsistencias: {(chk['channel_id_comment'] != chk['channel_id_video']).sum()} / {len(chk)}")

print()
print("5) ¿author_handle (sin '/') coincide con author_name (sin '@')?")
handle_norm = comments["author_handle"].str.lstrip("/").str.lstrip("@")
name_norm = comments["author_name"].str.lstrip("@")
mismatch_mask = handle_norm != name_norm
print(f"   Coincidencia: {(~mismatch_mask).mean():.1%}  ({mismatch_mask.sum()} inconsistencias de {len(comments)})")
print("   Ejemplos de inconsistencia (nombres con tilde/ñ codificados como URL-encoding en el handle):")
comments.loc[mismatch_mask, ["author_name", "author_handle"]].drop_duplicates().head(5)


1) ¿Un mismo channel_id tiene siempre el mismo channel_name?
   channel_id con más de un channel_name distinto: 0 / 97

2) ¿owner_handle coincide con channel_handle? (según el enunciado, deberían coincidir 100%)
   Coincidencia: 100.0%

3) ¿upload_date coincide con publish_date?
   Coincidencia: 100.0%

4) ¿El channel_id reportado en comments coincide con el channel_id del video (ya verificado en 1.3)?
   Inconsistencias: 0 / 406

5) ¿author_handle (sin '/') coincide con author_name (sin '@')?
   Coincidencia: 96.6%  (14 inconsistencias de 406)
   Ejemplos de inconsistencia (nombres con tilde/ñ codificados como URL-encoding en el handle):


,author_name,author_handle
73,@ErvinLeonardoCarreraLatín,/@ErvinLeonardoCarreraLat%C3%ADn
170,@AlejandroPérez-b6r,/@AlejandroP%C3%A9rez-b6r
230,@ErmePérez-q4s,/@ErmeP%C3%A9rez-q4s
251,@BorisTebalán,/@BorisTebal%C3%A1n
296,@RolandoCastroPérez-d7y,/@RolandoCastroP%C3%A9rez-d7y


### 2.2. Variables que no pueden utilizarse o requieren precauciones especiales

| Variable | Problema | Tratamiento / justificación |
|---|---|---|
| `viewer_rating` (comments) | 100% vacía (406/406) | Se descarta del análisis; no aporta información. |
| `is_pinned` (comments) | Constante (`False` en todos los registros) | Se conserva por completitud pero no se usa como variable explicativa. |
| `upload_date` (videos) | Idéntica a `publish_date` en el 100% de los casos | Redundante; se usa únicamente `publish_date`. |
| `owner_handle` (videos) | Idéntica a `channel_handle` en el 100% de los casos | Redundante; se usa únicamente `channel_handle`. |
| `channel_name`, `author_name` | Etiquetas visibles que pueden repetirse o cambiar en el tiempo | No se usan como identificador ni para hacer *joins*; solo como etiqueta al graficar. Se usan `channel_id` / `author_channel_id`. |
| `author_handle` | 3.4% de los casos con tildes/ñ codificadas como URL-encoding | Se genera `author_handle_decoded` únicamente para mostrar etiquetas; no se usa como identificador. |
| `view_count_text`, `like_count_text` | Numéricas almacenadas como texto (`"2,390 vistas"`, celdas en blanco) | Se convierten a numérico en 2.4; **no** se deben usar directamente en cálculos. |
| `view_count_text` | 13 valores faltantes (view_count no tiene faltantes) | Se prioriza `view_count` (más completo) para todo análisis cuantitativo, como recomienda el enunciado. |
| `description`, `description_snippet` | 25–26 valores faltantes; texto libre con URLs, hashtags, menciones y emojis | Se tratan como texto libre opcional; si se usan para tópicos requieren el mismo pipeline de limpieza que `text`. |
| `keywords`, `query_hits`, `dataset_sources` | Texto con estructura de lista (JSON-like) o valores separados por `\|` | Deben parsearse (`ast.literal_eval` / `str.split("\|")`) antes de analizarse; no son texto libre. |
| `reply_count` | Cuenta respuestas, no identifica a sus autores | **No debe interpretarse como arista entre usuarios** (aclarado explícitamente en el enunciado del laboratorio). |
| `text` (comments) | Contiene URLs, menciones, hashtags, emojis, texto en otros idiomas y un caso de emoji almacenado como texto literal (`:hand-purple-blue-peace:`) | Se preserva como `texto_original` (auditoría/sentimiento) y se deriva `texto_limpio` (seccs. 2.5–2.6). |


### 2.3. Normalización de identificadores y nombres


In [17]:
# IDs: se fuerzan a texto y se recortan espacios en blanco (sin alterar su valor)
id_cols_videos = ["video_id", "channel_id"]
id_cols_comments = ["video_id", "comment_id", "channel_id", "author_channel_id"]

for c in id_cols_videos:
    videos[c] = videos[c].astype(str).str.strip()
for c in id_cols_comments:
    comments[c] = comments[c].astype(str).str.strip()

# Nombres y handles: se recortan espacios, se preservan como etiquetas (no como identificadores)
for c in ["channel_name", "channel_handle", "owner_handle", "title"]:
    videos[c] = videos[c].astype(str).str.strip()
for c in ["author_name", "author_handle", "channel_name"]:
    comments[c] = comments[c].astype(str).str.strip()

# Handle decodificado solo para presentación (no reemplaza al identificador)
comments["author_handle_decoded"] = comments["author_handle"].apply(urllib.parse.unquote)

print("Ejemplo de decodificación de author_handle:")
comments.loc[
    comments["author_handle"] != comments["author_handle_decoded"],
    ["author_channel_id", "author_name", "author_handle", "author_handle_decoded"]
].drop_duplicates().head(5)


Ejemplo de decodificación de author_handle:


,author_channel_id,author_name,author_handle,author_handle_decoded
73,UC9vMq1YF-e7C-VO0SHkbUTA,@ErvinLeonardoCarreraLatín,/@ErvinLeonardoCarreraLat%C3%ADn,/@ErvinLeonardoCarreraLatín
170,UCtsxPYw8VUsH_FfnKg-0lmw,@AlejandroPérez-b6r,/@AlejandroP%C3%A9rez-b6r,/@AlejandroPérez-b6r
230,UCg3eIfd7BrsLEqS0BLkftHA,@ErmePérez-q4s,/@ErmeP%C3%A9rez-q4s,/@ErmePérez-q4s
251,UC1NiEdJ7gk6FlDexIalWnGg,@BorisTebalán,/@BorisTebal%C3%A1n,/@BorisTebalán
296,UCu8_3-Wg-YzNyvHYQDEscfw,@RolandoCastroPérez-d7y,/@RolandoCastroP%C3%A9rez-d7y,/@RolandoCastroPérez-d7y


### 2.4. Conversión de variables de conteo almacenadas como texto

- **`view_count_text`** (videos): formato `"2,390 vistas"`. Se elimina todo lo que no sea dígito (separador de
  miles `,` y el sufijo `" vistas"`). No se encontraron abreviaturas tipo `K`/`mil`/`M` en este campo.
- **`like_count_text`** (comments): valores puramente numéricos (`"4"`, `"191"`, …) o **celdas en blanco** (189 de
  406, 46.6%). Se asume que un valor en blanco corresponde a **0 "me gusta" mostrados** por YouTube — es la
  interpretación más razonable dado que YouTube oculta el contador cuando es muy bajo — y se documenta
  explícitamente como un supuesto, no como un hecho verificado.
- Se compara `view_count_text` (recién convertida) contra `view_count` ya numérica: existen pequeñas discrepancias
  en 53/280 videos con ambos valores disponibles, atribuibles a que ambos campos se capturaron en momentos distintos
  del proceso de recolección (no son errores de conversión). Esto confirma la recomendación del enunciado de usar
  `view_count` —más completa y estable— para los cálculos.


In [18]:
def parse_view_count_text(t):
    """'2,390 vistas' -> 2390 ; NaN -> NaN."""
    if pd.isna(t):
        return np.nan
    digitos = re.sub(r"[^\d]", "", str(t))
    return int(digitos) if digitos else np.nan

videos["view_count_text_parsed"] = videos["view_count_text"].apply(parse_view_count_text)

con_ambos = videos["view_count_text_parsed"].notna()
discrepancia = con_ambos & (videos["view_count_text_parsed"] != videos["view_count"])
print(f"Videos con view_count_text y view_count disponibles: {con_ambos.sum()}")
print(f"Discrepancias entre ambos: {discrepancia.sum()} ({discrepancia.sum()/con_ambos.sum():.1%})")
print(f"Videos con view_count_text faltante (view_count sigue disponible): {(~con_ambos).sum()}")
videos.loc[discrepancia, ["video_id", "view_count_text", "view_count_text_parsed", "view_count"]].head(5)


Videos con view_count_text y view_count disponibles: 280
Discrepancias entre ambos: 53 (18.9%)
Videos con view_count_text faltante (view_count sigue disponible): 13


,video_id,view_count_text,view_count_text_parsed,view_count
0,-5puKGEqcUc,"2,390 vistas",2390.0,2357
5,0LTLUY7LBlA,"6,098 vistas",6098.0,6099
6,0a4_g1R1-aA,"15,436 vistas",15436.0,15455
16,2iqOSL9YguA,"5,180 vistas",5180.0,5181
25,3onMfRym3OU,"4,485 vistas",4485.0,4484


In [19]:
def parse_like_count_text(t):
    """'' o solo espacios -> 0 (asumido); '191' -> 191."""
    s = str(t).strip()
    if s == "" or s.lower() == "nan":
        return 0
    digitos = re.sub(r"[^\d]", "", s)
    return int(digitos) if digitos else 0

comments["like_count"] = comments["like_count_text"].apply(parse_like_count_text)

n_blancos = (comments["like_count_text"].astype(str).str.strip() == "").sum()
print(f"like_count_text en blanco convertidos a 0 (supuesto documentado): {n_blancos} / {len(comments)}")
print()
print(comments["like_count"].describe())


like_count_text en blanco convertidos a 0 (supuesto documentado): 189 / 406

count    406.000000
mean       5.726601
std       30.661298
min        0.000000
25%        0.000000
50%        1.000000
75%        2.000000
max      405.000000
Name: like_count, dtype: float64


### 2.5. Texto original y texto limpio


In [20]:
comments["texto_original"] = comments["text"]


### 2.6. Pipeline de limpieza para `texto_limpio`

Antes de limpiar, se extraen a columnas separadas los elementos que se van a remover del texto:

- `hashtags`: palabras que siguen a `#`.
- `mentions`: nombres de usuario que siguen a `@`.
- `emojis`: emojis Unicode presentes en el comentario (vía la librería `emoji`).

Pasos aplicados sobre `texto_limpio`:

1. **Caracteres de ancho cero / `\xa0`**: se eliminan (aparecen en comentarios que empiezan citando una mención,
   p. ej. `"\u200b\xa0@usuario\xa0..."`) porque no son visibles pero rompen la tokenización.
2. **URLs**: se eliminan con una expresión regular (`http(s)://…`, `www.…`); solo 1 comentario contenía una URL.
3. **Emojis Unicode**: se eliminan del texto (ya quedaron capturados en la columna `emojis`).
4. **Emojis como texto literal**: se detectó que 2 comentarios contienen el emoji ya convertido a una etiqueta de
   texto por la fuente original (p. ej. `:hand-purple-blue-peace:`) en vez del carácter Unicode; se eliminan con
   una expresión regular adicional (`:palabra-palabra:`) porque el detector de emojis Unicode no los reconoce.
5. **Hashtags y menciones**: se eliminan del cuerpo del texto (ya quedaron capturados en columnas separadas).
6. **Minúsculas**: se convierte todo el texto a minúsculas.
7. **Puntuación**: se elimina todo carácter que no sea letra (incluyendo tildes/ñ), dígito o espacio.
8. **Números**: se eliminan los dígitos sueltos (no aportan al análisis de tópicos/frecuencias).
9. **Stopwords en español**: se eliminan usando la lista de `nltk` (313 palabras) y se descartan también tokens de
   un solo carácter.
10. **Lematización**: se aplica `simplemma` (lematizador ligero para español) a cada token restante, por ejemplo
    `"corruptos" → "corrupto"`, `"gobernó" → "gobernar"`.

**Limitación documentada:** el pipeline está calibrado para español. Los comentarios en otro idioma (se observan
algunos en inglés) no se ven beneficiados por la eliminación de stopwords ni por la lematización, ya que ambas
herramientas son específicas del español; esos textos quedan solo con las transformaciones genéricas
(minúsculas, URLs, puntuación, etc.).


In [21]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
HASHTAG_RE = re.compile(r"#(\w+)")
MENTION_RE = re.compile(r"@([A-Za-z0-9_\-.]+)")
TEXT_EMOJI_RE = re.compile(r":[a-z0-9]+(?:-[a-z0-9]+)*:")   # emoji guardado como texto literal, p.ej. :hand-blue:
PUNCT_RE = re.compile(r"[^\w\sñÑáéíóúÁÉÍÓÚüÜ]")
NUM_RE = re.compile(r"\b\d+\b")
MULTISPACE_RE = re.compile(r"\s+")
ZERO_WIDTH_RE = re.compile(r"[\u200b\xa0]")


def extraer_hashtags(t):
    return HASHTAG_RE.findall(str(t))


def extraer_menciones(t):
    return MENTION_RE.findall(str(t))


def extraer_emojis(t):
    return [c for c in str(t) if c in emoji.EMOJI_DATA]


def limpiar_texto(t):
    t = str(t)
    t = ZERO_WIDTH_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = emoji.replace_emoji(t, replace=" ")
    t = TEXT_EMOJI_RE.sub(" ", t)
    t = HASHTAG_RE.sub(" ", t)
    t = MENTION_RE.sub(" ", t)
    t = t.lower()
    t = PUNCT_RE.sub(" ", t)
    t = NUM_RE.sub(" ", t)
    t = MULTISPACE_RE.sub(" ", t).strip()
    tokens = [tok for tok in t.split() if tok not in STOPWORDS_ES and len(tok) > 1]
    tokens = [simplemma.lemmatize(tok, lang="es") for tok in tokens]
    return " ".join(tokens)


comments["hashtags"] = comments["texto_original"].apply(extraer_hashtags)
comments["mentions"] = comments["texto_original"].apply(extraer_menciones)
comments["emojis"] = comments["texto_original"].apply(extraer_emojis)
comments["texto_limpio"] = comments["texto_original"].apply(limpiar_texto)

comments[["texto_original", "texto_limpio", "hashtags", "mentions", "emojis"]].head(8)


,texto_original,texto_limpio,hashtags,mentions,emojis
0,Ese corrupto amigo de la vieja fiscal los teng...,corrupto amigo viejo fiscal verbose carcel,[],[],[]
1,"Están jóvenes porque no buscan un trabajo, tu...",joven buscar trabajo suerte policía gustar van...,[],[],[]
2,Me dejaron con ganas de demandar la ilegalidad...,dejar gana demandar ilegalidad reunión virtual...,[],[],[]
3,Veremos a este mafioso de Mazariegos en la cár...,verse mafioso mazariegos cárcel bueno tiempo s...,[],[],[]
4,eso es para que salga de USA por su propio pie...,salir usar propio pie auto deportar,[],[],[]
5,Imagine if they had to walk back home.,imaginar if they had to walk back home,[],[],[]
6,buenísima investigacion :hand-purple-blue-peac...,buenísima investigacion,[],[],[]
7,"Lleven su lonchera, sacrifiquense un poco. Y r...",llevar lonchera sacrifiquense reintevren diner...,[],[],[]


### 2.7. Efecto cuantificado de la limpieza

In [22]:
vacios_antes = (comments["texto_original"].astype(str).str.strip() == "").sum()
vacios_despues = (comments["texto_limpio"].astype(str).str.strip() == "").sum()
dup_antes = comments["texto_original"].duplicated().sum()
dup_despues = comments["texto_limpio"].duplicated().sum()
modificados = (comments["texto_original"].str.lower().str.strip() != comments["texto_limpio"]).sum()

print(f"Registros totales:                         {len(comments)} (ninguno se elimina en esta etapa)")
print(f"Textos modificados por la limpieza:         {modificados} / {len(comments)} ({modificados/len(comments):.1%})")
print(f"Textos vacíos ANTES de limpiar:              {vacios_antes}")
print(f"Textos vacíos DESPUÉS de limpiar:             {vacios_despues}")
print(f"Comentarios duplicados (texto exacto) ANTES:  {dup_antes}")
print(f"Comentarios duplicados (texto exacto) DESPUÉS: {dup_despues}")


Registros totales:                         406 (ninguno se elimina en esta etapa)
Textos modificados por la limpieza:         396 / 406 (97.5%)
Textos vacíos ANTES de limpiar:              0
Textos vacíos DESPUÉS de limpiar:             6
Comentarios duplicados (texto exacto) ANTES:  2
Comentarios duplicados (texto exacto) DESPUÉS: 11


In [23]:
print("Comentarios que quedaron vacíos tras la limpieza (dominados por emojis/símbolos sin contenido léxico):")
comments.loc[comments["texto_limpio"].astype(str).str.strip() == "", ["comment_id", "texto_original"]]


Comentarios que quedaron vacíos tras la limpieza (dominados por emojis/símbolos sin contenido léxico):


,comment_id,texto_original
38,UgwIlv8gtF3WKYfHSJJ4AaABAg,😮
49,UgwSDn5aIUEZwOlx9c94AaABAg,❤🎉
97,UgwxjmtBoAozs0ZUAcd4AaABAg.AaAbqoGVGipAaAjNG9oPRN,que?
129,UgxEE17R1iQyuDZebVF4AaABAg.A_UFk_pMAbMA_VZ_0Vdu89,😂😂
139,UgxEsAK7q3l34wFdZ414AaABAg.A_zCnf5bSajAa0qEPC8tJc,​ @Murmullodelbarrio 👍🇬🇹
187,UgxoLjEjkngSVGZvO1N4AaABAg,👏👏👏👏👏👏


**Interpretación:**
- El **97.5%** de los comentarios (396/406) se modificó de alguna forma, lo cual es esperable dado el número de transformaciones aplicadas.
- **6 comentarios (1.5%)** quedaron vacíos tras la limpieza: todos estaban compuestos únicamente por emojis,
  puntuación o una mención sin más contenido. Estos registros
  **no se eliminan** del `DataFrame` — siguen siendo válidos para el conteo de participación y para el análisis de
  sentimiento con `texto_original`; pero deben excluirse de los análisis basados en
  palabras/bigramas de `texto_limpio`, ya que no aportan tokens.
- Los **duplicados de texto exacto pasaron de 2 a 11** después de la limpieza. Esto no indica un error: al
  normalizar minúsculas, puntuación y lematizar, comentarios que eran ligeramente distintos en su forma pero
  equivalentes en contenido terminan colapsando al mismo texto limpio.
  Para el análisis de contenido (nube de palabras, frecuencias, bigramas) esto es deseable; para conteos de
  participación se debe seguir usando `comment_id` como unidad, no `texto_limpio`.
